In [1]:
# =============================================================
# Notebook : xBD-03b-Crop-Hold
# Step 1 (Cell 1): Setup and load the SAME crop config as xBD-03
# -------------------------------------------------------------
# Purpose : Locate image_index / building_index (phase-1 output)
#           and crop_config.json (xBD-03 output). Reusing the
#           saved config guarantees hold crops follow exactly
#           the same rules as the train/val crops.
# Change  : only roles_in_this_notebook -> ["hold"]
# Output  : /kaggle/working/crop_config.json (hold copy)
# =============================================================

import os
import json
import time
import numpy as np
import pandas as pd

WORK_DIR = "/kaggle/working"

def find_file(name, root="/kaggle/input", skip_dirs=("xbd-dataset",)):
    """Search /kaggle/input for a file, skipping the huge xBD image folders."""
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        if name in filenames:
            return os.path.join(dirpath, name)
    raise FileNotFoundError(f"{name} not found - check the notebook inputs")

PATHS = {
    "image_index":    find_file("image_index.csv"),
    "building_index": find_file("building_index.parquet"),
    "crop_config":    find_file("crop_config.json"),
}
for k, v in PATHS.items():
    print(f"{k:15s}: {v}")

image_index = pd.read_csv(PATHS["image_index"], dtype={"img_id": str})
assert os.path.exists(image_index["pre_img"].iloc[0]), "Add qianlanzz/xbd-dataset as input"

with open(PATHS["crop_config"]) as f:
    cfg = json.load(f)
cfg["roles_in_this_notebook"] = ["hold"]          # the ONLY change
OUT = cfg["out_size"]

with open(os.path.join(WORK_DIR, "crop_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

print("\nLoaded crop settings:")
for k in ["min_side_px", "context_scale", "min_crop_px", "max_crop_px",
          "out_size", "pad_value", "image_format", "mask_format", "shard_size"]:
    print(f"   {k:15s}= {cfg[k]}")

image_index    : /kaggle/input/notebooks/fahimratul/phase-1-data-preparation/image_index.csv
building_index : /kaggle/input/notebooks/fahimratul/phase-1-data-preparation/building_index.parquet
crop_config    : /kaggle/input/notebooks/fahimratul/phase-3-crop-dataset/crop_config.json

Loaded crop settings:
   min_side_px    = 8
   context_scale  = 1.5
   min_crop_px    = 64
   max_crop_px    = 1024
   out_size       = 128
   pad_value      = 0
   image_format   = npy_uint8
   mask_format    = packbits
   shard_size     = 20000


In [2]:
# =============================================================
# Notebook : xBD-03b-Crop-Hold
# Step 2 (Cell 2): Filter hold buildings and plan the crops
# -------------------------------------------------------------
# Purpose : Keep ALL labelled hold buildings (0-3) with longest
#           side >= 8 px - no sampling, so the final evaluation
#           uses the real class distribution. Crop windows are
#           computed with exactly the same formula as xBD-03.
# Output  : /kaggle/working/crop_plan.parquet
# =============================================================

bld = pd.read_parquet(PATHS["building_index"])
bld["longest"] = bld[["bbox_w", "bbox_h"]].max(axis=1)
plan = bld[(bld["role"] == "hold")
           & (bld["label"] >= 0)
           & (bld["longest"] >= cfg["min_side_px"])].copy()

# ---- Crop geometry (same formula as xBD-03) ----
plan["cx"] = (plan["x_min"] + plan["x_max"]) / 2
plan["cy"] = (plan["y_min"] + plan["y_max"]) / 2
side = np.ceil(plan["longest"] * cfg["context_scale"])
side = side.clip(lower=cfg["min_crop_px"], upper=cfg["max_crop_px"])
plan["crop_side"] = side.astype(int)
plan["crop_x0"]   = np.floor(plan["cx"] - plan["crop_side"] / 2).astype(int)
plan["crop_y0"]   = np.floor(plan["cy"] - plan["crop_side"] / 2).astype(int)
plan["needs_pad"] = (
    (plan["crop_x0"] < 0) | (plan["crop_y0"] < 0)
    | (plan["crop_x0"] + plan["crop_side"] > 1024)
    | (plan["crop_y0"] + plan["crop_side"] > 1024)
)

plan = plan.sort_values(["key", "uid"]).reset_index(drop=True)
plan["sample_id"] = np.arange(len(plan))

# ---- Summary (counts only - hold stays "sealed") ----
print(f"Hold buildings: {len(plan):,}")
print("Label counts  :", plan["label"].value_counts().sort_index().to_dict())
print("Images to read:", plan["key"].nunique())
gb = len(plan) * cfg["bytes_per_sample"] / 1e9
print(f"Raw storage   : {gb:.2f} GB in {int(np.ceil(len(plan) / cfg['shard_size']))} shard(s)")

plan_cols = ["sample_id", "role", "split", "key", "disaster", "uid", "subtype",
             "label", "x_min", "y_min", "x_max", "y_max", "longest", "area",
             "cx", "cy", "crop_side", "crop_x0", "crop_y0", "needs_pad", "wkt"]
plan = plan[plan_cols]
plan.to_parquet(os.path.join(WORK_DIR, "crop_plan.parquet"), index=False)
print("Saved crop_plan.parquet")

Hold buildings: 52,838
Label counts  : {0: 37754, 1: 6307, 2: 4600, 3: 4177}
Images to read: 751
Raw storage   : 5.30 GB in 3 shard(s)
Saved crop_plan.parquet


In [3]:
# =============================================================
# Notebook : xBD-03b-Crop-Hold
# Step 3 (Cell 3): Cropping functions (identical to xBD-03)
# -------------------------------------------------------------
# Purpose : Same functions as xBD-03 Cell 3:
#           crop_with_pad -> resize_img -> target_mask ->
#           process_image (one image pair read once, all its
#           buildings cropped, masks bit-packed).
# Output  : Functions only (no files)
# =============================================================

import cv2
from shapely import wkt as shp_wkt
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

key_to_paths = image_index.set_index("key")[["pre_img", "post_img"]].to_dict("index")

def read_rgb(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise IOError(f"Cannot read {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def crop_with_pad(img, x0, y0, side, pad_value=0):
    H, W = img.shape[:2]
    out = np.full((side, side, img.shape[2]), pad_value, dtype=img.dtype)
    xs, ys = max(x0, 0), max(y0, 0)
    xe, ye = min(x0 + side, W), min(y0 + side, H)
    if xe > xs and ye > ys:
        out[ys - y0:ye - y0, xs - x0:xe - x0] = img[ys:ye, xs:xe]
    return out

def resize_img(crop):
    interp = cv2.INTER_AREA if crop.shape[0] > OUT else cv2.INTER_LINEAR
    return cv2.resize(crop, (OUT, OUT), interpolation=interp)

def target_mask(wkt_str, x0, y0, side):
    scale = OUT / side
    mask = np.zeros((OUT, OUT), dtype=np.uint8)
    geom = shp_wkt.loads(wkt_str)
    parts = geom.geoms if geom.geom_type == "MultiPolygon" else [geom]
    for g in parts:
        pts = (np.asarray(g.exterior.coords) - [x0, y0]) * scale
        cv2.fillPoly(mask, [np.round(pts).astype(np.int32)], 1)
    return mask

def process_image(args):
    key, rows = args
    paths = key_to_paths[key]
    pre, post = read_rgb(paths["pre_img"]), read_rgb(paths["post_img"])
    results = []
    for r in rows.itertuples(index=False):
        pre_c  = resize_img(crop_with_pad(pre,  r.crop_x0, r.crop_y0, r.crop_side, cfg["pad_value"]))
        post_c = resize_img(crop_with_pad(post, r.crop_x0, r.crop_y0, r.crop_side, cfg["pad_value"]))
        mask   = target_mask(r.wkt, r.crop_x0, r.crop_y0, r.crop_side)
        results.append((r.pos, pre_c, post_c, np.packbits(mask.reshape(-1)), int(mask.sum())))
    return results

print("Functions ready")

Functions ready


In [4]:
# =============================================================
# Notebook : xBD-03b-Crop-Hold
# Step 4 (Cell 4): Crop all hold buildings and write raw shards
# -------------------------------------------------------------
# Purpose : Same writing logic as xBD-03 Cell 4:
#           pre-created memmap .npy shards, 16 threads, each
#           sample written straight into its slot, then
#           hold_index.parquet + hold_DONE flag.
# Output  : /kaggle/working/crops/hold_XX_{pre,post,mask}.npy
#           /kaggle/working/crops/hold_index.parquet
# =============================================================

from numpy.lib.format import open_memmap

CROP_DIR  = os.path.join(WORK_DIR, "crops")
os.makedirs(CROP_DIR, exist_ok=True)
SHARD     = cfg["shard_size"]
N_WORKERS = 16
BATCH     = 64

for role in cfg["roles_in_this_notebook"]:
    done_flag = os.path.join(CROP_DIR, f"{role}_DONE")
    if os.path.exists(done_flag):
        print(f"[{role}] already done - skipped")
        continue

    df = plan.copy()
    df["pos"]    = np.arange(len(df))
    df["shard"]  = df["pos"] // SHARD
    df["offset"] = df["pos"] % SHARD
    n_shards = int(df["shard"].max()) + 1

    shards = []
    for s in range(n_shards):
        n = int((df["shard"] == s).sum())
        base = os.path.join(CROP_DIR, f"{role}_{s:02d}")
        shards.append({
            "pre":  open_memmap(base + "_pre.npy",  mode="w+", dtype=np.uint8, shape=(n, OUT, OUT, 3)),
            "post": open_memmap(base + "_post.npy", mode="w+", dtype=np.uint8, shape=(n, OUT, OUT, 3)),
            "mask": open_memmap(base + "_mask.npy", mode="w+", dtype=np.uint8, shape=(n, OUT * OUT // 8)),
        })

    mask_pixels = np.zeros(len(df), dtype=np.int32)
    groups = list(df.groupby("key", sort=False))
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex, \
         tqdm(total=len(groups), desc=f"Cropping {role}") as pbar:
        for i in range(0, len(groups), BATCH):
            for results in ex.map(process_image, groups[i:i + BATCH]):
                for pos, pre_c, post_c, mpack, mpix in results:
                    s, o = divmod(pos, SHARD)
                    shards[s]["pre"][o]  = pre_c
                    shards[s]["post"][o] = post_c
                    shards[s]["mask"][o] = mpack
                    mask_pixels[pos] = mpix
                pbar.update(1)

    for sh in shards:
        for arr in sh.values():
            arr.flush()
    del shards

    df["mask_pixels"] = mask_pixels
    df.to_parquet(os.path.join(CROP_DIR, f"{role}_index.parquet"), index=False)
    with open(done_flag, "w") as f:
        f.write("ok")
    print(f"[{role}] {len(df):,} samples in {n_shards} shard(s) "
          f"- {(time.time() - t0) / 60:.1f} min")

total = sum(os.path.getsize(os.path.join(CROP_DIR, f)) for f in os.listdir(CROP_DIR))
print(f"\nTotal size of crops/: {total / 1e9:.2f} GB")

Cropping hold:   0%|          | 0/751 [00:00<?, ?it/s]

[hold] 52,838 samples in 3 shard(s) - 1.2 min

Total size of crops/: 5.32 GB


In [5]:
# =============================================================
# Notebook : xBD-03b-Crop-Hold
# Step 5 (Cell 5): Verify hold shards (numbers only, no images)
# -------------------------------------------------------------
# Purpose : Reload shards read-only and check label counts,
#           empty masks and all-black crops. No samples are
#           displayed - the hold set stays "sealed" until
#           the final evaluation in Phase 7.
# Output  : Printed checks
# =============================================================

idx = pd.read_parquet(os.path.join(CROP_DIR, "hold_index.parquet"))
n_shards = int(idx["shard"].max()) + 1
arrays = [{k: np.load(os.path.join(CROP_DIR, f"hold_{s:02d}_{k}.npy"), mmap_mode="r")
           for k in ("pre", "post", "mask")} for s in range(n_shards)]

print(f"hold: {len(idx):,} samples in {n_shards} shard(s)")
print("Label counts:", idx["label"].value_counts().sort_index().to_dict())
print(f"Empty target masks: {(idx['mask_pixels'] == 0).sum():,}")
print("Mask pixels (median / 5% / 95%):",
      idx["mask_pixels"].median(),
      idx["mask_pixels"].quantile(0.05),
      idx["mask_pixels"].quantile(0.95))

chk = idx.sample(500, random_state=0)
black = sum(arrays[r.shard]["post"][r.offset].max() == 0 for r in chk.itertuples())
print(f"All-black post crops in 500 random samples: {black}")

hold: 52,838 samples in 3 shard(s)
Label counts: {0: 37754, 1: 6307, 2: 4600, 3: 4177}
Empty target masks: 0
Mask pixels (median / 5% / 95%): 2451.0 422.0 4756.0
All-black post crops in 500 random samples: 0
